# 2.4 Prompting Techniques & Patterns

## Playground Notebook

This notebook covers the **core prompting strategies** that turn a generic LLM into a precise, reliable tool.

| Technique | Core Idea |
|-----------|----------|
| **Zero-shot** | Ask directly — no examples needed |
| **One-shot / Few-shot** | Teach by showing 1 or more examples |
| **Role-based** | Assign a persona to shape tone, depth, and perspective |
| **Structured Output** | Force responses into JSON, XML, Markdown, etc. |
| **Prompt Templates** | Reusable patterns with placeholders for different inputs |

---

In [1]:
import json
import time
from IPython.display import display, Markdown, HTML
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# ============================================================
#  CONFIGURATION - Change the model name here if needed
# ============================================================
MODEL = "qwen2.5:1.5b"

llm = ChatOllama(model=MODEL)

# ============================================================
#  HELPER FUNCTIONS
# ============================================================

def build_messages(message_dicts):
    """Convert a list of role/content dicts into LangChain message objects."""
    type_map = {
        "system": SystemMessage,
        "user": HumanMessage,
        "assistant": AIMessage,
    }
    return [type_map[m["role"]](content=m["content"]) for m in message_dicts]


def chat(messages, **kwargs):
    """Send messages to the model and return the response text."""
    _llm = ChatOllama(model=MODEL, **kwargs) if kwargs else llm
    lc_messages = build_messages(messages)
    start = time.time()
    response = _llm.invoke(lc_messages)
    elapsed = time.time() - start
    content = response.content
    display(Markdown(content))
    print(f"\n⏱️ {elapsed:.2f}s | {len(content)} chars")
    return content


def show_messages(messages):
    """Pretty-print the message list being sent to the model."""
    colors = {"system": "#e74c3c", "user": "#3498db", "assistant": "#2ecc71"}
    html = ""
    for msg in messages:
        role = msg["role"]
        color = colors.get(role, "#888")
        content_preview = msg["content"][:300] + ("..." if len(msg["content"]) > 300 else "")
        html += (
            f'<div style="margin:6px 0;padding:8px 12px;border-left:4px solid {color};'
            f'background:#1e1e1e;border-radius:4px;">'
            f'<strong style="color:{color};text-transform:uppercase;">{role}</strong>'
            f'<br><span style="color:#ccc;white-space:pre-wrap;">{content_preview}</span></div>'
        )
    display(HTML(html))


print(f"✅ Using model: {MODEL}")

c:\Users\shiva\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ Using model: qwen2.5:1.5b


---

## 1. Zero-Shot Prompting — No Examples Needed

In **zero-shot** prompting, you give the model a task with **no examples** at all. You rely entirely on the model's pre-trained knowledge and clear instructions.

```
Zero-shot = Instruction only, no demonstrations
```

This works well when:
- The task is straightforward (classification, summarization, translation)
- The model already understands the format you want
- You want quick results without crafting examples

### Experiment 1A: Basic Zero-Shot Tasks

In [2]:
zero_shot_tasks = [
    {
        "label": "Sentiment Classification",
        "prompt": "Classify the sentiment of this review as Positive, Negative, or Neutral.\n\nReview: \"The battery life is incredible but the screen is too dim outdoors.\""
    },
    {
        "label": "Translation",
        "prompt": "Translate the following English sentence to Spanish:\n\n\"The weather is beautiful today and I want to go for a walk.\""
    },
    {
        "label": "Summarization",
        "prompt": "Summarize the following paragraph in one sentence:\n\n\"Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing computer programs that can access data and use it to learn for themselves. The process begins with observations or data, such as examples, direct experience, or instruction, in order to look for patterns in data and make better decisions in the future.\""
    }
]

for task in zero_shot_tasks:
    print(f"\n{'=' * 60}")
    print(f"  ZERO-SHOT: {task['label']}")
    print(f"{'=' * 60}")
    messages = [{"role": "user", "content": task["prompt"]}]
    show_messages(messages)
    _ = chat(messages, temperature=0.0)


  ZERO-SHOT: Sentiment Classification


To classify the sentiment of the given review, we need to analyze the key phrases and overall tone:

1. **"The battery life is incredible"** - This phrase indicates a positive aspect of the product.
2. **"but the screen is too dim outdoors"** - This phrase suggests a negative aspect related to functionality.

Overall, while there are two parts to the review, both contribute positively to the overall sentiment. The first part highlights an excellent feature (battery life), and the second part points out a minor issue that doesn't significantly impact the user's experience negatively.

Therefore, based on this analysis, the sentiment of the review is **Positive**.


⏱️ 3.31s | 670 chars

  ZERO-SHOT: Translation


"El clima es hermoso hoy y quiero hacer una caminata."


⏱️ 0.29s | 54 chars

  ZERO-SHOT: Summarization


Machine learning is a branch of artificial intelligence that allows systems to automatically improve through experience without explicit programming.


⏱️ 0.32s | 149 chars


### Experiment 1B: When Zero-Shot Struggles

Zero-shot can fail on **ambiguous, nuanced, or domain-specific** tasks where the model doesn't know the exact format you expect.

In [5]:
# A task where zero-shot may give inconsistent formatting
prompt = """Extract the product name, price, and currency from this text:

"I just bought the Sony WH-1000XM5 headphones for €349.99 at the airport."
"""

print("ZERO-SHOT EXTRACTION (no format guidance)")
print("=" * 60)
messages = [{"role": "user", "content": prompt}]
show_messages(messages)
result_zs = chat(messages, temperature=0.7)

print("\n💡 Notice: The model returns an answer, but the format is unpredictable.")
print("   We'll fix this with few-shot examples later!")

ZERO-SHOT EXTRACTION (no format guidance)


Product Name: Sony WH-1000XM5 headphones
Price: €349.99
Currency: Euro (EUR)


⏱️ 0.65s | 76 chars

💡 Notice: The model returns an answer, but the format is unpredictable.
   We'll fix this with few-shot examples later!


---

## 2. One-Shot & Few-Shot Prompting — Teaching by Example

Instead of just *telling* the model what to do, you *show* it with examples:

```
Zero-shot:  "Classify this sentiment"            → 0 examples
One-shot:   "Here's one example, now do this"     → 1 example
Few-shot:   "Here are 3 examples, now do this"    → 2-5 examples
```

Examples are provided as **user → assistant** message pairs in the conversation history.

### Experiment 2A: Zero-Shot vs. One-Shot vs. Few-Shot

In [6]:
# ---- ZERO-SHOT ----
print("=" * 60)
print("  ZERO-SHOT (0 examples)")
print("=" * 60)

messages_zero = [
    {"role": "system", "content": "Classify the emotion in each sentence. Reply with the word: Happy, Sad, Angry, or Fearful."},
    {"role": "user", "content": "I can't believe they cancelled the concert after I waited all year."}
]
show_messages(messages_zero)
_ = chat(messages_zero, temperature=0.7)

  ZERO-SHOT (0 examples)


Sad


⏱️ 0.33s | 3 chars


In [8]:
# ---- ONE-SHOT ----
print("=" * 60)
print("  ONE-SHOT (1 example)")
print("=" * 60)

messages_one = [
    {"role": "system", "content": "Classify the emotion in each sentence. Reply with exactly one word: Happy, Sad, Angry, or Fearful."},
    {"role": "user", "content": "I just got promoted at work!"},
    {"role": "assistant", "content": "Happy"},
    {"role": "user", "content": "I can't believe they cancelled the concert after I waited all year."}
]
show_messages(messages_one)
_ = chat(messages_one, temperature=0.0)

  ONE-SHOT (1 example)


Sad


⏱️ 0.30s | 3 chars


In [ ]:
# ---- FEW-SHOT ----
print("=" * 60)
print("  FEW-SHOT (3 examples)")
print("=" * 60)

messages_few = [
    {"role": "system", "content": "Classify the emotion in each sentence. Reply with exactly one word: Happy, Sad, Angry, or Fearful."},
    # Example 1
    {"role": "user", "content": "I just got promoted at work!"},
    {"role": "assistant", "content": "Happy"},
    # Example 2
    {"role": "user", "content": "My dog passed away last night."},
    {"role": "assistant", "content": "Sad"},
    # Example 3
    {"role": "user", "content": "There's a strange noise coming from the basement."},
    {"role": "assistant", "content": "Fearful"},
    # Actual task
    {"role": "user", "content": "I can't believe they cancelled the concert after I waited all year."}
]
show_messages(messages_few)
_ = chat(messages_few, temperature=0.0)

### Experiment 2B: Few-Shot Fixes Formatting Problems

Remember the extraction task that had unpredictable formatting in zero-shot? Let's fix it with examples.

In [11]:
messages = [
    {"role": "system", "content": "Extract product name, price, and currency from the text. Use the exact format shown in the examples."},
    # Example 1
    {"role": "user", "content": "I purchased the Apple MacBook Air for $1,299 from the online store."},
    {"role": "assistant", "content": "Product: Apple MacBook Air Price: 1299 Currency: USD"},
    # Example 2
    {"role": "user", "content": "The Samsung Galaxy S24 was on sale for £799 at Heathrow."},
    {"role": "assistant", "content": "Product: Samsung Galaxy S24 Price: 799 Currency: GBP"},
    # Actual task
    {"role": "user", "content": "I just bought the Sony WH-1000XM5 headphones for €349.99 at the airport."}
]

print("FEW-SHOT EXTRACTION (2 examples → consistent format)")
print("=" * 60)
show_messages(messages)
_ = chat(messages, temperature=0.0)

print("\n💡 Compare this to the zero-shot result above — the format is now consistent!")

FEW-SHOT EXTRACTION (2 examples → consistent format)


Product: Sony WH-1000XM5 Price: 349.99 Currency: EUR


⏱️ 0.60s | 52 chars

💡 Compare this to the zero-shot result above — the format is now consistent!


### Experiment 2C: Few-Shot for Custom Classification

Few-shot is especially powerful for **custom categories** that don't exist in standard training data.

In [12]:
# Custom support ticket classification — categories specific to a fictional company
messages = [
    {"role": "system", "content": "Classify each support ticket into one category: BILLING, BUG, FEATURE_REQUEST, or ACCOUNT_ACCESS. Reply with the category only."},
    # Examples teach the model our specific category definitions
    {"role": "user", "content": "I was charged twice for my subscription this month."},
    {"role": "assistant", "content": "BILLING"},
    {"role": "user", "content": "The export button crashes the app on Safari."},
    {"role": "assistant", "content": "BUG"},
    {"role": "user", "content": "It would be great if you added dark mode."},
    {"role": "assistant", "content": "FEATURE_REQUEST"},
    {"role": "user", "content": "I forgot my password and the reset email never arrived."},
    {"role": "assistant", "content": "ACCOUNT_ACCESS"},
    # New tickets to classify
    {"role": "user", "content": "Can you add the ability to schedule reports weekly?"}
]

print("FEW-SHOT CUSTOM CLASSIFICATION")
print("=" * 60)
show_messages(messages)
_ = chat(messages, temperature=0.0)

# Test with another ticket
messages[-1] = {"role": "user", "content": "My invoice shows the wrong company name."}
print("\n--- Next ticket ---")
_ = chat(messages, temperature=0.0)

messages[-1] = {"role": "user", "content": "The page goes blank when I click on my profile settings."}
print("\n--- Next ticket ---")
_ = chat(messages, temperature=0.0)

FEW-SHOT CUSTOM CLASSIFICATION


FEATURE_REQUEST


⏱️ 0.40s | 15 chars

--- Next ticket ---


BILLING


⏱️ 0.18s | 7 chars

--- Next ticket ---


BUG


⏱️ 0.16s | 3 chars


---

## 3. Role-Based Prompting — Assigning Personas

By assigning a **role** or **persona** via the system prompt, you control:
- **Expertise level** — beginner-friendly vs. expert-level depth
- **Tone** — formal, casual, enthusiastic, clinical
- **Perspective** — what the model prioritizes and how it frames answers
- **Constraints** — what the model should and shouldn't discuss

### Experiment 3A: Same Question, Different Expert Roles

In [7]:
question = "Why is my Python program running slowly?"

code = """

def calculate_average(grades):
    total = 0
    for i in range(len(grades)):
        total = total + grades[i]
    average = total / len(grades)
    return average

def get_grade(average):
    if average >= 90:
        return "A"
    if average >= 80:
        return "B"
    if average >= 70:
        return "C"
    if average >= 60:
        return "D"
    else:
        return "F"

def print_report(name, grades):
    average = calculate_average(grades)
    grade = get_grade(average)
    print("Student: " + name)
    print("Grades: " + grades)
    print("Average: " + average)
    print("Grade: " + grade)

students = {
    "Alice": [85, 90, 78],
    "Bob":   [70, 65, 80],
    "Charlie": []
}

for name, grades in students.items():
    print_report(name, grades)
    print("---")


"""

roles = [
    {
        "label": "Junior Developer Mentor",
        "system": "You are a patient senior developer mentoring a junior. Use simple language, avoid jargon, and give practical first steps. Keep it to 3-4 bullet points."
    },
    {
        "label": "Performance Engineer",
        "system": "You are a performance engineer specializing in Python optimization. Be technical and specific — mention profiling tools, algorithmic complexity, and memory management. Keep it to 3-4 bullet points."
    },
    {
        "label": "Sarcastic Code Reviewer",
        "system": "You are a brutally honest but funny code reviewer. Point out common mistakes with dry humor. Keep it to 3-4 bullet points."
    }
]

for role in roles:
    print(f"\n{'=' * 60}")
    print(f"  ROLE: {role['label']}")
    print(f"{'=' * 60}")
    messages = [
        {"role": "system", "content": role["system"]},
        {"role": "user", "content": question + code}
    ]
    show_messages(messages)
    _ = chat(messages, temperature=0.7)


  ROLE: Junior Developer Mentor


To optimize your Python program and make it run faster, consider the following practical steps:

1. **Use List Comprehensions Instead of Loops**: In this case, using a list comprehension for calculating the average can be more efficient.
2. **Avoid Redundant Operations**: Ensure that all operations are performed on the same data structure to avoid unnecessary copying or recalculations.

Here’s how you can refactor your code:

```python
def calculate_average(grades):
    return sum(grades) / len(grades)

def get_grade(average):
    if average >= 90:
        return "A"
    elif average >= 80:
        return "B"
    elif average >= 70:
        return "C"
    elif average >= 60:
        return "D"
    else:
        return "F"

def print_report(name, grades):
    average = calculate_average(grades)
    grade = get_grade(average)
    print(f"Student: {name}")
    print("Grades: " + str(grades))
    print(f"Average: {average:.2f}")
    print(f"Grade: {grade}")

students = {
    "Alice": [85, 90, 78],
    "Bob": [70, 65, 80],
    "Charlie": []
}

for name, grades in students.items():
    print_report(name, grades)
```

### Explanation:
1. **Calculate Average**:
   - Instead of using a loop and `total` variable to sum the list elements, we use a single line with `sum(grades) / len(grades)` which is more concise and efficient.

2. **Get Grade**:
   - The conditionals were simplified by combining multiple conditions into a single `elif` statement for better readability.

3. **Formatting Strings**:
   - Added formatting to the average number for two decimal places using `.2f`.

4. **Print Report**:
   - Removed redundant use of `print("Average: " + average)` and replaced it with a formatted string output, which is easier to read and more efficient.

By following these steps, your program should run faster due to reduced redundancy and simpler code structure.


⏱️ 6.98s | 1878 chars

  ROLE: Performance Engineer


To optimize your Python program and potentially make it run faster, consider the following strategies:

1. **Use Efficient Algorithms**: The provided `calculate_average` function can be optimized by removing unnecessary operations within the loop and using list comprehensions for better readability and performance. Also, considering that you're dividing a total sum of grades by their count in each iteration, this operation could be simplified to just return the accumulated sum divided by its length directly.

   ```python
   def calculate_average(grades):
       return sum(grades) / len(grades)
   ```

2. **Profile Your Code**: Use profiling tools such as `cProfile` or `line_profiler` to identify bottlenecks in your code. Look for sections where the program spends significant time, focusing on areas that could be optimized.

3. **Optimize Memory Management**: Ensure you're not performing unnecessary memory operations. For instance, using list comprehension inside a loop can create multiple lists unnecessarily, which is less efficient than creating them once and then accessing elements by index.

   ```python
   students = {
       "Alice": [85, 90, 78],
       "Bob":   [70, 65, 80],
       "Charlie": []
   }

   for name, grades in students.items():
       print_report(name, grades)
       average = calculate_average(grades)  # Directly compute the average instead of summing
       grade = get_grade(int(average))      # Ensure the average is an integer to avoid float issues
       print("Grade: " + grade)
   ```

By applying these optimizations, your program should run faster. Remember that profiling tools can provide detailed insights into where improvements are most needed, and it’s often a good idea to test different approaches on small sample data before scaling up to larger datasets or more complex scenarios.


⏱️ 4.44s | 1845 chars

  ROLE: Sarcastic Code Reviewer


1. The program is unnecessarily complex for simple tasks.
2. There's a missing `return` statement at the end of `get_grade`.
3. Instead of passing a string to `print`, you should format it correctly using f-strings or `.format()`.


⏱️ 0.76s | 230 chars


### Experiment 3B: Role-Based for Audience Adaptation

The same topic explained to **different audiences** using role prompts.

In [8]:
topic = "Explain what an API is."

audiences = [
    {
        "label": "For a 10-year-old",
        "system": "You are a fun science teacher at a primary school. Explain concepts using everyday analogies a 10-year-old would understand. Keep it to 2-3 sentences."
    },
    {
        "label": "For a business executive",
        "system": "You are a technology consultant presenting to a non-technical CEO. Focus on business value, not implementation details. Keep it to 2-3 sentences."
    },
    {
        "label": "For a software engineer",
        "system": "You are a senior backend architect. Be precise and technical. Mention protocols, patterns, and relevant standards. Keep it to 2-3 sentences."
    }
]

for aud in audiences:
    print(f"\n{'=' * 60}")
    print(f"  AUDIENCE: {aud['label']}")
    print(f"{'=' * 60}")
    messages = [
        {"role": "system", "content": aud["system"]},
        {"role": "user", "content": topic}
    ]
    show_messages(messages)
    _ = chat(messages, temperature=0.5)


  AUDIENCE: For a 10-year-old


An API, or Application Programming Interface, is like a set of instructions that tells computers how to talk to each other and share information in the digital world. It's kind of like when you tell your friends how to play a game - it's all about sharing rules and making sure everyone knows what they're doing. Just as games need clear rules for players to understand, APIs provide clear directions so that different programs can communicate with each other seamlessly on the internet.


⏱️ 1.35s | 487 chars

  AUDIENCE: For a business executive


An API (Application Programming Interface) is like a set of rules and protocols that allows different software applications to talk to each other and share information or perform actions on behalf of the user. It's a way for one application to communicate with another without being directly connected, which can be useful in creating complex systems where data needs to flow between multiple components seamlessly.


⏱️ 0.85s | 415 chars

  AUDIENCE: For a software engineer


An Application Programming Interface (API) is a set of rules and protocols that allows different software applications to communicate with each other. It defines the methods by which one application can request services from another, enabling integration between unrelated systems for tasks such as data manipulation, authentication, or business logic execution. APIs are crucial in facilitating interoperability among various technologies and platforms within an ecosystem.


⏱️ 0.90s | 474 chars


### Experiment 3C: Role + Few-Shot Combined

The most effective prompts often **combine techniques**. Here we pair a role with few-shot examples.

In [13]:
messages = [
    {
        "role": "system",
        "content": (
            "You are a medical triage assistant at a hospital reception. "
            "Based on the patient's symptoms, assign a triage level: "
            "GREEN (non-urgent), YELLOW (semi-urgent), or RED (emergency). "
            "Format: LEVEL — brief reason (one sentence)."
        )
    },
    # Few-shot examples
    {"role": "user", "content": "Patient has a mild headache that started this morning."},
    {"role": "assistant", "content": "GREEN — Mild headache with no other symptoms suggests a non-urgent condition."},
    {"role": "user", "content": "Patient reports chest pain radiating to the left arm with shortness of breath."},
    {"role": "assistant", "content": "RED — Chest pain with arm radiation and dyspnea indicates possible myocardial infarction."},
    {"role": "user", "content": "Patient has a deep cut on the forearm, bleeding is controlled with pressure."},
    {"role": "assistant", "content": "YELLOW — Deep laceration needs medical attention but bleeding is currently controlled."},
    # Actual cases
    {"role": "user", "content": "Patient has a high fever of 39.5°C, stiff neck, and sensitivity to light."}
]

print("ROLE + FEW-SHOT: Medical Triage Assistant")
print("=" * 60)
show_messages(messages)
_ = chat(messages, temperature=0.0)

# Try another case
messages[-1] = {"role": "user", "content": "Patient has a runny nose and occasional sneezing for 2 days."}
print("\n--- Next patient ---")
_ = chat(messages, temperature=0.0)

ROLE + FEW-SHOT: Medical Triage Assistant


RED — High fever with stiff neck and photophobia suggests meningitis, an emergency condition.


⏱️ 0.70s | 93 chars

--- Next patient ---


GREEN — Runny nose and occasional sneezing for 2 days suggests a viral upper respiratory infection.


⏱️ 0.35s | 99 chars


---

## 4. Structured Output Generation

Getting the model to respond in a **specific format** (JSON, XML, Markdown) is critical for building reliable pipelines. Three strategies, in order of reliability:

1. **Zero-shot with format instructions** — tell the model what format you want
2. **Few-shot with format examples** — show the exact format you expect
3. **System prompt + few-shot** — combine role constraints with examples

### Experiment 4A: JSON Output — Zero-Shot vs. Few-Shot

In [9]:
input_text = "Marie Curie was born in Warsaw in 1867. She won two Nobel Prizes — one in Physics (1903) and one in Chemistry (1911)."

# ---- Zero-shot JSON ----
print("=" * 60)
print("  ZERO-SHOT JSON")
print("=" * 60)

messages_zs = [
    {"role": "system", "content": "Extract information from the text and respond with valid JSON only. No extra text. Keys: name, birthplace, birth_year, achievements (list of objects with field and year)."},
    {"role": "user", "content": input_text}
]
show_messages(messages_zs)
result_zs = chat(messages_zs, temperature=0.0)

try:
    json.loads(result_zs)
    print("✅ Valid JSON")
except json.JSONDecodeError:
    print("❌ Invalid JSON")

  ZERO-SHOT JSON


{
  "name": "Marie Curie",
  "birthplace": "Warsaw",
  "birth_year": 1867,
  "achievements": [
    {
      "and_year": 1903
    },
    {
      "and_year": 1911
    }
  ]
}


⏱️ 1.04s | 171 chars
✅ Valid JSON


In [10]:
# ---- Few-shot JSON (more reliable) ----
print("=" * 60)
print("  FEW-SHOT JSON")
print("=" * 60)

messages_fs = [
    {"role": "system", "content": "Extract information from the text. Respond with valid JSON only."},
    # Example
    {"role": "user", "content": "Albert Einstein was born in Ulm in 1879. He won the Nobel Prize in Physics in 1921."},
    {"role": "assistant", "content": json.dumps({
        "name": "Albert Einstein",
        "birthplace": "Ulm",
        "birth_year": 1879,
        "achievements": [{"field": "Physics", "year": 1921}]
    }, indent=2)},
    # Actual task
    {"role": "user", "content": input_text}
]
show_messages(messages_fs)
result_fs = chat(messages_fs, temperature=0.0)

try:
    parsed = json.loads(result_fs)
    print("✅ Valid JSON")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError:
    print("❌ Invalid JSON")

  FEW-SHOT JSON


{
  "name": "Marie Curie",
  "birthplace": "Warsaw",
  "birth_year": 1867,
  "achievements": [
    {
      "field": "Physics",
      "year": 1903
    },
    {
      "field": "Chemistry",
      "year": 1911
    }
  ]
}


⏱️ 1.37s | 217 chars
✅ Valid JSON
{
  "name": "Marie Curie",
  "birthplace": "Warsaw",
  "birth_year": 1867,
  "achievements": [
    {
      "field": "Physics",
      "year": 1903
    },
    {
      "field": "Chemistry",
      "year": 1911
    }
  ]
}


### Experiment 4B: XML Output

In [11]:
messages = [
    {"role": "system", "content": "Convert the book information into XML format. Respond with valid XML only."},
    # Example
    {"role": "user", "content": "The Great Gatsby by F. Scott Fitzgerald, published 1925, genre: Fiction"},
    {"role": "assistant", "content": """<book>
  <title>The Great Gatsby</title>
  <author>F. Scott Fitzgerald</author>
  <year>1925</year>
  <genre>Fiction</genre>
</book>"""},
    # Actual task
    {"role": "user", "content": "A Brief History of Time by Stephen Hawking, published 1988, genre: Science"}
]

print("FEW-SHOT XML GENERATION")
print("=" * 60)
show_messages(messages)
_ = chat(messages, temperature=0.0)

FEW-SHOT XML GENERATION


<book>
  <title>A Brief History of Time</title>
  <author>Stephen Hawking</author>
  <year>1988</year>
  <genre>Science</genre>
</book>


⏱️ 0.88s | 135 chars


### Experiment 4C: Markdown Table Output

In [12]:
messages = [
    {
        "role": "system",
        "content": "Convert the raw data into a well-formatted Markdown table. Include a header row. Respond with only the Markdown table."
    },
    # Example
    {"role": "user", "content": "Languages: Python - 1991 - Guido van Rossum; JavaScript - 1995 - Brendan Eich"},
    {"role": "assistant", "content": """| Language | Year | Creator |
|----------|------|---------|
| Python | 1991 | Guido van Rossum |
| JavaScript | 1995 | Brendan Eich |"""},
    # Actual task
    {"role": "user", "content": "Frameworks: React - 2013 - Meta; Angular - 2016 - Google; Vue.js - 2014 - Evan You; Svelte - 2016 - Rich Harris"}
]

print("FEW-SHOT MARKDOWN TABLE")
print("=" * 60)
show_messages(messages)
result = chat(messages, temperature=0.0)

print("\n📋 Rendered table:")
display(Markdown(result))

FEW-SHOT MARKDOWN TABLE


| Framework | Year | Company |
|-----------|------|----------|
| React | 2013 | Meta    |
| Angular | 2016 | Google   |
| Vue.js | 2014 | Evan You  |
| Svelte | 2016 | Rich Harris |


⏱️ 1.12s | 181 chars

📋 Rendered table:


| Framework | Year | Company |
|-----------|------|----------|
| React | 2013 | Meta    |
| Angular | 2016 | Google   |
| Vue.js | 2014 | Evan You  |
| Svelte | 2016 | Rich Harris |

### Experiment 4D: Multiple Formats from the Same Data

Let's extract the same information in **three different formats** to see how format instructions change the output.

In [13]:
raw_data = """Company: Acme Corp. Revenue: $5.2M. Employees: 120. Founded: 2018. Headquarters: Austin, TX."""

formats = [
    {
        "label": "JSON",
        "system": "Extract company data from the text. Respond with valid JSON only.",
        "example_in": "Company: FooBar Inc. Revenue: $1M. Employees: 30. Founded: 2020. Headquarters: NYC.",
        "example_out": json.dumps({"company": "FooBar Inc.", "revenue": "$1M", "employees": 30, "founded": 2020, "hq": "NYC"}, indent=2)
    },
    {
        "label": "XML",
        "system": "Extract company data from the text. Respond with valid XML only.",
        "example_in": "Company: FooBar Inc. Revenue: $1M. Employees: 30. Founded: 2020. Headquarters: NYC.",
        "example_out": "<company>\n  <name>FooBar Inc.</name>\n  <revenue>$1M</revenue>\n  <employees>30</employees>\n  <founded>2020</founded>\n  <hq>NYC</hq>\n</company>"
    },
    {
        "label": "Markdown Key-Value",
        "system": "Extract company data from the text. Respond as a Markdown bullet list with bold keys.",
        "example_in": "Company: FooBar Inc. Revenue: $1M. Employees: 30. Founded: 2020. Headquarters: NYC.",
        "example_out": "- **Company:** FooBar Inc.\n- **Revenue:** $1M\n- **Employees:** 30\n- **Founded:** 2020\n- **HQ:** NYC"
    }
]

for fmt in formats:
    print(f"\n{'=' * 60}")
    print(f"  FORMAT: {fmt['label']}")
    print(f"{'=' * 60}")
    messages = [
        {"role": "system", "content": fmt["system"]},
        {"role": "user", "content": fmt["example_in"]},
        {"role": "assistant", "content": fmt["example_out"]},
        {"role": "user", "content": raw_data}
    ]
    _ = chat(messages, temperature=0.0)


  FORMAT: JSON


{
  "company": "Acme Corp.",
  "revenue": "$5.2M",
  "employees": 120,
  "founded": 2018,
  "hq": "Austin, TX"
}


⏱️ 0.89s | 112 chars

  FORMAT: XML


<company>
  <name>Acme Corp.</name>
  <revenue>$5.2M</revenue>
  <employees>120</employees>
  <founded>2018</founded>
  <hq>Austin, TX</hq>
</company>


⏱️ 0.78s | 150 chars

  FORMAT: Markdown Key-Value


- **Company:** Acme Corp.
- **Revenue:** $5.2M
- **Employees:** 120
- **Founded:** 2018
- **HQ:** Austin, TX


⏱️ 0.59s | 108 chars


---

## 5. Prompt Templates & Reusability Patterns

Instead of writing a new prompt from scratch every time, build **reusable templates** with placeholders. This makes your prompts:
- **Consistent** across inputs
- **Testable** with different data
- **Maintainable** as a single source of truth

### Experiment 5A: Basic Template with String Formatting

In [14]:
# A reusable template for product descriptions
PRODUCT_TEMPLATE = """Write a compelling product description for an e-commerce listing.

Product: {product_name}
Category: {category}
Key Features: {features}
Target Audience: {audience}

Requirements:
- Write exactly 2-3 sentences
- Highlight the main benefit first
- Include a call to action"""

# Use the template with different products
products = [
    {
        "product_name": "AeroFit Pro Running Shoes",
        "category": "Athletic Footwear",
        "features": "Lightweight mesh, carbon-fiber plate, 30mm stack height",
        "audience": "Competitive marathon runners"
    },
    {
        "product_name": "ZenBrew Smart Kettle",
        "category": "Kitchen Appliances",
        "features": "Temperature presets, app control, keep-warm function",
        "audience": "Tea and coffee enthusiasts"
    }
]

for product in products:
    print(f"\n{'=' * 60}")
    print(f"  PRODUCT: {product['product_name']}")
    print(f"{'=' * 60}")
    prompt = PRODUCT_TEMPLATE.format(**product)
    messages = [{"role": "user", "content": prompt}]
    show_messages(messages)
    _ = chat(messages, temperature=0.7)


  PRODUCT: AeroFit Pro Running Shoes


Introducing the AeroFit Pro Running Shoes, designed for competitive marathon runners seeking the perfect blend of performance and comfort. With a lightweight mesh upper and a carbon-fiber plate, these shoes offer unmatched speed and durability. The 30mm stack height ensures effortless stride, perfect for pushing through the toughest runs. Elevate your training and finish races with the AeroFit Pro, where every step is a step towards achieving your marathon dreams. Lace up now and transform your next race.


⏱️ 1.49s | 510 chars

  PRODUCT: ZenBrew Smart Kettle


Introducing the ZenBrew Smart Kettle, the ultimate companion for your tea and coffee enthusiasts! With its intuitive temperature presets, you can easily brew the perfect cup of tea or coffee every time. Additionally, the app control allows you to monitor and adjust your brews from anywhere, ensuring your perfect brews are always within reach. Say goodbye to guesswork and hello to the ZenBrew Smart Kettle - your new best friend in the kitchen!


⏱️ 1.13s | 446 chars


### Experiment 5B: Template with System + User Separation

A well-designed template separates the **behavior** (system) from the **data** (user).

In [ ]:
# Reusable template as a function
def code_review_prompt(code_snippet, language="Python"):
    """Generate a code review using a standard template."""
    return [
        {
            "role": "system",
            "content": (
                f"You are a senior {language} code reviewer. "
                "For every review, respond with exactly this format:\n"
                "**Summary:** (one sentence)\n"
                "**Issues:** (numbered list or 'None')\n"
                "**Suggestions:** (numbered list or 'None')\n"
                "**Rating:** (1-5 stars)"
            )
        },
        {
            "role": "user",
            "content": f"Review this {language} code:\n\n```{language.lower()}\n{code_snippet}\n```"
        }
    ]


# Use the template for different code snippets
snippets = [
    {
        "label": "Python — Questionable function",
        "lang": "Python",
        "code": """def process(data):
    result = []
    for i in range(len(data)):
        if data[i] != None:
            result.append(data[i] * 2)
    return result"""
    },
    {
        "label": "JavaScript — Clean function",
        "lang": "JavaScript",
        "code": """const filterAdults = (people) => 
  people.filter(person => person.age >= 18)
        .map(({ name, age }) => ({ name, age }));"""
    }
]

for snippet in snippets:
    print(f"\n{'=' * 60}")
    print(f"  {snippet['label']}")
    print(f"{'=' * 60}")
    messages = code_review_prompt(snippet["code"], snippet["lang"])
    show_messages(messages)
    _ = chat(messages, temperature=0.3)

### Experiment 5C: Template Library — Multiple Reusable Patterns

In production, you maintain a **library of templates** for different tasks.

In [ ]:
# A mini template library
TEMPLATES = {
    "summarize": {
        "system": "You are a concise summarizer. Summarize the text in exactly {length}. No extra commentary.",
        "user": "Summarize this:\n\n{text}"
    },
    "translate": {
        "system": "You are a professional translator. Translate accurately while preserving tone and meaning.",
        "user": "Translate the following from {source_lang} to {target_lang}:\n\n{text}"
    },
    "explain_like": {
        "system": "You are an expert teacher. Explain the concept at the level appropriate for the audience. Keep it to 2-3 sentences.",
        "user": "Explain '{concept}' as if I am {audience}."
    }
}


def use_template(template_name, **kwargs):
    """Build messages from a template in the library."""
    tmpl = TEMPLATES[template_name]
    return [
        {"role": "system", "content": tmpl["system"].format(**kwargs)},
        {"role": "user", "content": tmpl["user"].format(**kwargs)}
    ]


# --- Use the summarize template ---
print("=" * 60)
print("  TEMPLATE: summarize")
print("=" * 60)
messages = use_template(
    "summarize",
    length="one sentence",
    text="Artificial intelligence has transformed industries from healthcare to finance. Machine learning models can now diagnose diseases, predict market trends, and automate complex workflows that previously required human expertise."
)
show_messages(messages)
_ = chat(messages, temperature=0.3)

In [ ]:
# --- Use the translate template ---
print("=" * 60)
print("  TEMPLATE: translate")
print("=" * 60)
messages = use_template(
    "translate",
    source_lang="English",
    target_lang="French",
    text="The best way to predict the future is to create it."
)
show_messages(messages)
_ = chat(messages, temperature=0.3)

In [ ]:
# --- Use the explain_like template ---
print("=" * 60)
print("  TEMPLATE: explain_like")
print("=" * 60)
messages = use_template(
    "explain_like",
    concept="blockchain",
    audience="a 12-year-old who loves video games"
)
show_messages(messages)
_ = chat(messages, temperature=0.5)

# Same template, different audience
print("\n" + "=" * 60)
print("  TEMPLATE: explain_like (different audience)")
print("=" * 60)
messages = use_template(
    "explain_like",
    concept="blockchain",
    audience="a bank executive evaluating new technology investments"
)
show_messages(messages)
_ = chat(messages, temperature=0.5)

---

## 6. Sandbox — Try It Yourself!

Combine any techniques from this notebook.

In [ ]:
# ============================================================
#  SANDBOX - Mix and match techniques!
# ============================================================

messages = [
    # Try: role-based system prompt
    {"role": "system", "content": "You are a helpful assistant. Respond in JSON format."},

    # Try: few-shot examples (uncomment to add)
    # {"role": "user", "content": "your example input"},
    # {"role": "assistant", "content": "your example output"},

    # Your actual prompt
    {"role": "user", "content": "What are 3 benefits of exercise?"}
]

print("YOUR CUSTOM EXPERIMENT")
print("=" * 60)
show_messages(messages)
_ = chat(messages, temperature=0.5)

---

## Key Takeaways

| Technique | When to Use | Reliability |
|-----------|-------------|-------------|
| **Zero-shot** | Simple, well-known tasks (translation, summarization) | Medium — format can vary |
| **One-shot** | When one example is enough to show the pattern | Good |
| **Few-shot** | Custom formats, domain-specific tasks, consistent output | High |
| **Role-based** | Controlling tone, expertise level, and perspective | High |
| **Structured output** | Pipelines, APIs, downstream processing | High with few-shot |
| **Templates** | Reusable tasks across many inputs | High — best for production |

### Combining Techniques — The Power Stack

```
┌─────────────────────────────────────────┐
│  SYSTEM: Role + format constraints      │  ← Role-based
├─────────────────────────────────────────┤
│  USER:   Example input 1                │
│  ASST:   Example output 1               │  ← Few-shot
│  USER:   Example input 2                │
│  ASST:   Example output 2               │
├─────────────────────────────────────────┤
│  USER:   Template.format(actual_data)   │  ← Template
├─────────────────────────────────────────┤
│  ASST:   Structured, consistent output  │  ← Reliable result
└─────────────────────────────────────────┘
```